# NovelReader T4 — tự tải GitHub + xuất Google Drive
1. Chọn **Runtime → Change runtime type → T4 GPU**.
2. Chạy từ bước 1: notebook tự tải ZIP tại đường dẫn GitHub bạn cung cấp, **không cần chọn/tải ZIP thủ công**.
3. Notebook kèm lớp cập nhật v4 cho xuất Drive, dùng được cả khi ZIP trên GitHub còn là v3.
4. Bấm **Xuất TXT / EPUB / Audio**, chọn nơi lưu **Google Drive**. Nếu chưa kết nối, **quay về tab Colab để cấp quyền** khi được hỏi; không nhập token vào Web.

Mỗi phiên có link Web riêng được bảo vệ bằng cookie sau khi mở link trong notebook. Không chia sẻ link đó.
Chỉ khi bạn bấm xuất lên Drive, notebook mới yêu cầu `drive.mount`. Quyền mount của Colab có phạm vi rộng hơn chỉ thư mục xuất; chỉ cấp quyền cho notebook/mã nguồn bạn tin cậy.
Kết quả được sao chép vào `MyDrive/NovelReader_Exports/<tên truyện>/`; bản gốc trên Colab được giữ nguyên.
Không cam kết tốc độ T4; dữ liệu `/content` là tạm, cần sao lưu trước khi ngắt phiên.

## 1. Tự tải ZIP từ GitHub và cập nhật mã nguồn


In [ ]:
import os, sys, subprocess, zipfile, stat, urllib.request, hashlib, json, base64, zlib
from pathlib import Path
from google.colab import files

REPO_DIR = Path('/content/NovelReader_T4_Web')
SOURCE_URL = 'https://raw.githubusercontent.com/dungbeommm/NovelReader_Translate/main/NovelReader_GPU_notebook.zip'
for name in ('server_process', 'tunnel_process'):
    proc = globals().get(name)
    if proc is not None and proc.poll() is None:
        raise RuntimeError('Dừng cell Web/tunnel trước khi tải/cập nhật mã nguồn.')
archive = Path('/content/NovelReader_GPU_notebook.zip')
partial = archive.with_suffix('.zip.part')
print('Đang tải:', SOURCE_URL)
try:
    with urllib.request.urlopen(SOURCE_URL, timeout=90) as src, partial.open('wb') as out:
        total = 0
        while True:
            chunk = src.read(1024 * 1024)
            if not chunk:
                break
            total += len(chunk)
            if total > 128 * 1024 * 1024:
                raise RuntimeError('ZIP mã nguồn vượt giới hạn 128 MB.')
            out.write(chunk)
    if not zipfile.is_zipfile(partial):
        raise RuntimeError('Đường dẫn GitHub không trả về ZIP hợp lệ. Kiểm tra file còn public.')
    os.replace(partial, archive)
finally:
    partial.unlink(missing_ok=True)
print('ZIP SHA256:', hashlib.sha256(archive.read_bytes()).hexdigest())
with zipfile.ZipFile(archive) as z:
    infos = z.infolist()
    if len(infos) > 5000 or sum(i.file_size for i in infos) > 512 * 1024 * 1024:
        raise RuntimeError('ZIP có quá nhiều tệp hoặc dung lượng giải nén quá lớn.')
    markers = [n for n in z.namelist() if n == 'main.py' or (n.endswith('/main.py') and len(Path(n).parts) == 2)]
    if len(markers) != 1:
        raise RuntimeError('Không xác định được thư mục dự án trong ZIP.')
    prefix = markers[0][:-len('main.py')]
    for info in infos:
        if not info.filename.startswith(prefix):
            continue
        rel = info.filename[len(prefix):]
        if not rel:
            continue
        target = (REPO_DIR / rel).resolve()
        if not target.is_relative_to(REPO_DIR.resolve()) or stat.S_ISLNK(info.external_attr >> 16):
            raise RuntimeError('ZIP có đường dẫn hoặc symlink không an toàn.')
        # Updating code must never replace user's data, weights, or credentials.
        if Path(rel).parts[0] in {'data', 'models', 'output_text', 'output_audio', '.git'} or Path(rel).name == '.env':
            continue
        if info.is_dir():
            target.mkdir(parents=True, exist_ok=True)
        else:
            target.parent.mkdir(parents=True, exist_ok=True)
            with z.open(info) as src, target.open('wb') as dst:
                import shutil
                shutil.copyfileobj(src, dst)
# Source-only overlay maintains v4 functionality even if GitHub currently serves v3.
OVERLAY_V4 = '...'
for rel, content in json.loads(zlib.decompress(base64.b64decode(OVERLAY_V4))).items():
    path = REPO_DIR / rel
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding='utf-8')
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print('Đã nạp bản Web có xuất Drive. Thư mục:', REPO_DIR)
subprocess.run(['nvidia-smi'], check=True)

## 2. Cài thư viện (không thay PyTorch CUDA có sẵn của Colab)
Chỉ dùng CT2 nên không cần tải lại PyTorch. Bước này cài requirements trước, sau đó gỡ hai bản ORT và chỉ cài lại ORT GPU.
Nếu trước đó đã import ONNX Runtime/Piper/CT2 trong kernel, hãy **Restart session**, rồi chạy lại từ bước 1. Không đổi runtime sang CPU.

In [ ]:
if any(module in sys.modules for module in ('onnxruntime', 'piper', 'ctranslate2')):
    raise RuntimeError('Đã import GPU runtime. Restart session trước khi cài lại để tránh giữ thư viện cũ trong bộ nhớ.')
subprocess.run([sys.executable, 'scripts/setup_colab_runtime.py'], check=True)
subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'ffmpeg', 'espeak-ng'], check=True)

## 3. Cấu hình T4
Web mặc định batch **96** (cho phép chỉnh đến 256), CT2 nội bộ **96**, beam **2**. Thử beam 1 trong Web nếu chấp nhận đánh đổi chất lượng.
Pipeline audio có công tắc trong Web để so sánh với chế độ tuần tự. Nếu VRAM thiếu, giảm batch 96 → 64 → 32.
Piper CUDA là bắt buộc ở notebook này, không chuyển CPU âm thầm. Để chủ động thử CPU TTS: đặt PIPER_DEVICE=cpu và PIPER_REQUIRE_CUDA=0, khởi động lại server; smoke check bước 4 khi đó sẽ không phù hợp vì nó kiểm tra CUDA bắt buộc.

In [ ]:
import site
os.environ.update({
    'HACHIMIMT_WEB_BATCH_SIZE': '96',
    'HACHIMIMT_BATCH_SIZE': '96',
    'HACHIMIMT_CT2_BATCH_TYPE': 'tokens',
    'HACHIMIMT_CT2_WINDOW_MULTIPLIER': '16',
    'HACHIMIMT_INTER_THREADS': '1',
    'HACHIMIMT_COMPUTE_TYPE': 'int8_float16',
    'HACHIMIMT_FORCE_CT2_CUDA': '1',
    'HACHIMIMT_GPU_INDICES': '0',
    'HACHIMIMT_AUTO_ALL_GPUS': '0',
    'HACHIMIMT_TOKENIZE_WORKERS': str(min(os.cpu_count() or 2, 4)),
    'PIPER_DEVICE': 'cuda', 'PIPER_REQUIRE_CUDA': '1',
    'ALLOWED_HOSTS': '*', 'ALLOW_TUNNEL': '1', 'COLAB_GPU': '1',
    'NOVEL_DATA': str(REPO_DIR / 'data'),
    'NOVEL_MODELS': str(REPO_DIR / 'models/translation'),
    'TTS_VOICE_DIR': str(REPO_DIR / 'models/voices'),
    'NOVEL_OUTPUT_TEXT': str(REPO_DIR / 'output_text'),
    'NOVEL_OUTPUT_AUDIO': str(REPO_DIR / 'output_audio'),
    'PYTHONUNBUFFERED': '1',
    'NOVEL_COLAB_DRIVE_BRIDGE': '1',
    'NOVEL_COLAB_DRIVE_ROOT': '/content/drive/MyDrive',
})
# Export NVIDIA wheel libraries before starting child processes.
libdirs = []
for base in site.getsitepackages():
    libdirs.extend(str(p) for p in (Path(base) / 'nvidia').glob('*/lib') if p.is_dir())
os.environ['LD_LIBRARY_PATH'] = ':'.join(libdirs + [os.environ.get('LD_LIBRARY_PATH', '')])
for name in ('data/logs', 'models/translation', 'models/voices', 'output_text', 'output_audio'):
    (REPO_DIR / name).mkdir(parents=True, exist_ok=True)
print('Đã đặt cấu hình. Chạy bước 4 rồi bước 5.')

## 4. Tải model cần dùng và kiểm tra GPU thật
Chỉ tải giọng mặc định, không tải toàn bộ kho giọng. Script dịch một câu và tạo WAV ngắn; nó dừng nếu CT2/Piper không dùng CUDA.
Nếu bước này lỗi, xem thông báo CUDA/cuDNN; đừng tiếp tục chạy Web rồi tưởng là đã dùng GPU.

In [ ]:
subprocess.run([sys.executable, 'scripts/check_gpu.py',
                '--model', 'HachimiMT-60-QT', '--voice', 'ngochuyennew'], check=True)

## 5. Khởi động Web + cấp quyền Drive khi bấm xuất
Cell này phải tiếp tục chạy để phục vụ Web và hiển thị yêu cầu cấp quyền Drive.
- Link Web là link riêng theo phiên, mở một lần sẽ đổi thành cookie HttpOnly; không chia sẻ link.
- Trong Web chọn **Lưu: Google Drive** rồi bấm xuất. Nếu chưa mount, quay lại **tab Colab này** để cho phép Drive; Web không tự tạo được hộp thoại Google của Colab trong tab tunnel.
- Chọn **Lưu: Tải về máy** nếu không muốn cấp quyền Drive.
- Không dừng cell khi đang sao chép file lớn lên Drive. Stop khi đã có thông báo thành công; dữ liệu gốc vẫn nằm ở `/content`.

In [ ]:
import time, re, urllib.request, signal, secrets
from google.colab import drive
from gdrive.colab_bridge import process_drive_requests
from IPython.display import display, Markdown

for name in ('server_process', 'tunnel_process'):
    old = globals().get(name)
    if old is not None and old.poll() is None:
        raise RuntimeError('Server/tunnel cũ còn chạy. Dừng cell cũ trước.')
# Fail early if port 8000 is already occupied.
import socket
with socket.socket() as sock:
    sock.bind(('127.0.0.1', 8000))
cloudflared = Path('/content/cloudflared')
if not cloudflared.is_file():
    urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', cloudflared)
    cloudflared.chmod(0o755)
# Per-session secret is never written to the ZIP or Drive.
os.environ['NOVEL_WEB_ACCESS_TOKEN'] = secrets.token_urlsafe(32)
os.environ['NOVEL_EXPORT_SESSION_ID'] = secrets.token_hex(16)
server_log = REPO_DIR / 'data/logs/colab-server.log'
tunnel_log = REPO_DIR / 'data/logs/colab-tunnel.log'
server_process = tunnel_process = None
try:
    with server_log.open('w') as sf, tunnel_log.open('w') as tf:
        server_process = subprocess.Popen(
            [sys.executable, '-m', 'uvicorn', 'main:app', '--host', '127.0.0.1',
             '--port', '8000', '--workers', '1', '--no-access-log', '--timeout-graceful-shutdown', '120'],
            cwd=REPO_DIR, stdout=sf, stderr=subprocess.STDOUT)
        for _ in range(120):
            if server_process.poll() is not None:
                raise RuntimeError(server_log.read_text()[-6000:])
            try:
                with urllib.request.urlopen(urllib.request.Request('http://127.0.0.1:8000/api/state', headers={'X-Novel-Access': os.environ['NOVEL_WEB_ACCESS_TOKEN']}), timeout=2):
                    break
            except Exception:
                time.sleep(1)
        else:
            raise RuntimeError('Web chưa sẵn sàng. Xem colab-server.log.')
        tunnel_process = subprocess.Popen(
            [str(cloudflared), 'tunnel', '--url', 'http://127.0.0.1:8000'],
            stdout=tf, stderr=subprocess.STDOUT)
        public_url = None
        for _ in range(90):
            if tunnel_process.poll() is not None:
                raise RuntimeError(tunnel_log.read_text()[-4000:])
            match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', tunnel_log.read_text())
            if match:
                public_url = match.group(0)
                break
            time.sleep(1)
        if not public_url:
            raise RuntimeError('Không lấy được URL tunnel. Xem colab-tunnel.log.')
        display(Markdown(f"### [Mở NovelReader]({public_url}/?access_token={os.environ['NOVEL_WEB_ACCESS_TOKEN']})\nLink riêng của phiên — không chia sẻ. Khi bấm xuất Drive, quay lại tab này để cấp quyền nếu được hỏi."))
        with server_log.open() as log:
            while server_process.poll() is None and tunnel_process.poll() is None:
                process_drive_requests(REPO_DIR / 'data/drive_exports', drive.mount)
                line = log.readline()
                if line:
                    print(line, end='')
                else:
                    time.sleep(.2)
except KeyboardInterrupt:
    print('Đang dừng. Không đóng runtime cho tới khi hoàn tất...')
finally:
    for proc in (tunnel_process, server_process):
        if proc is not None and proc.poll() is None:
            proc.terminate()
    for proc in (tunnel_process, server_process):
        if proc is not None and proc.poll() is None:
            try:
                proc.wait(timeout=135)
            except subprocess.TimeoutExpired:
                proc.kill()
                proc.wait()
                print('Đã buộc dừng quá thời hạn; audio chưa xong sẽ được làm lại khi tiếp tục.')
    print('Đã dừng Web và tunnel.')

## 6. Tải kết quả sau khi dừng Web
Gói gồm text/audio và database checkpoint. Không đưa token/cấu hình Google Drive vào bản sao lưu này. Sao lưu chỉ nhất quán khi server đã dừng.

In [ ]:
if globals().get('server_process') is not None and server_process.poll() is None:
    raise RuntimeError('Dừng cell Web trước khi đóng gói checkpoint.')
from datetime import datetime
backup = Path('/content') / ('NovelReader_outputs_' + datetime.now().strftime('%Y%m%d_%H%M%S') + '.zip')
with zipfile.ZipFile(backup, 'w', zipfile.ZIP_DEFLATED) as z:
    for folder in ('output_text', 'output_audio'):
        for p in (REPO_DIR / folder).rglob('*'):
            if p.is_file() and not any(part.startswith('.') for part in p.relative_to(REPO_DIR).parts):
                z.write(p, p.relative_to(REPO_DIR))
    # SQLite backup API also includes committed WAL data.
    import sqlite3, tempfile
    db = REPO_DIR / 'data/novel.sqlite3'
    if db.exists():
        with tempfile.TemporaryDirectory() as temp:
            snapshot = Path(temp) / 'novel.sqlite3'
            with sqlite3.connect(db) as src, sqlite3.connect(snapshot) as dst:
                src.backup(dst)
            z.write(snapshot, 'data/novel.sqlite3')
print('Bản sao lưu:', backup.name)
files.download(str(backup))

## Đo tốc độ (tùy chọn, dừng Web trước)
Dùng `translation/core/benchmark_sweep.py` với cùng file TXT để so batch/beam/compute type. Đây là benchmark core dịch, không đại diện thời gian Web + TTS.
Trong log Web có `TIMING translation` và `TIMING audio` theo chương. So sánh cùng vài chương, cùng giọng, cùng thiết lập để tránh kết luận sai.